# PaySim: Basic Exploratory Data Analysis

Explore transaction characteristics, data quality, and fraud patterns in the PaySim synthetic financial dataset.

**Objectives**
- Inspect the dataset structure, missing values, and duplicate records.
- Summarize transaction types, amounts, and activity over time.
- Compare fraudulent and non-fraudulent transactions.
- Examine the existing fraud flag and identify next steps.

Run the cells from top to bottom. The full CSV is approximately 494 MB, so allow sufficient memory (several GB). This notebook loads all rows; only the scatter plot uses a sample.


In [1]:
# Run this cell if the required packages are not installed.
%pip install pandas numpy matplotlib


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (9, 5), 'axes.grid': True})
RANDOM_STATE = 42


## 1. Load the Dataset

The path below supports running from the project root or a notebook subdirectory. Update it if you move the CSV elsewhere.

`step` represents one simulated hour. `isFraud` is the fraud label; `isFlaggedFraud` is an existing system flag. Account identifiers are categorical identifiers, not numeric measurements. Merchant accounts can have zero destination balances because those balances are not recorded in PaySim; interpret these values with care.


In [3]:
candidates = [Path('data/paysim dataset.csv'), Path('../data/paysim dataset.csv')]
DATA_PATH = next((path for path in candidates if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Place paysim dataset.csv in the project data directory.')

dtypes = {
    'step': 'int32', 'type': 'category',
    'nameOrig': 'string', 'nameDest': 'string',
    'isFraud': 'int8', 'isFlaggedFraud': 'int8',
}
df = pd.read_csv(DATA_PATH, dtype=dtypes)
print(f'Loaded {len(df):,} rows and {df.shape[1]} columns.')
print(f'DataFrame memory: {df.memory_usage(deep=True).sum() / 1024**2:,.1f} MB')
display(df.head())


Loaded 6,362,620 rows and 11 columns.


DataFrame memory: 1,104.1 MB


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,"9,839.6400",C1231006815,"170,136.0000","160,296.3600",M1979787155,0.0000,0.0000,0,0
1,1,PAYMENT,"1,864.2800",C1666544295,"21,249.0000","19,384.7200",M2044282225,0.0000,0.0000,0,0
2,1,TRANSFER,181.0000,C1305486145,181.0000,0.0000,C553264065,0.0000,0.0000,1,0
3,1,CASH_OUT,181.0000,C840083671,181.0000,0.0000,C38997010,"21,182.0000",0.0000,1,0
4,1,PAYMENT,"11,668.1400",C2048537720,"41,554.0000","29,885.8600",M1230701703,0.0000,0.0000,0,0


## 2. Structure and Data Quality

Check column types, missing values, exact duplicate rows, and unexpected values before interpreting patterns. Duplicate rows are reported without automatically removing them.

In [4]:
df.info()
display(df.describe(include='number').T)
quality = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_percent': df.isna().mean() * 100,
})
display(quality)
print(f'Exact duplicate rows: {df.duplicated().sum():,}')
print(f'Negative transaction amounts: {(df["amount"] < 0).sum():,}')
for column in ['isFraud', 'isFlaggedFraud']:
    print(f'{column} values: {sorted(df[column].dropna().unique().tolist())}')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype   
---  ------          -----   
 0   step            int32   
 1   type            category
 2   amount          float64 
 3   nameOrig        string  
 4   oldbalanceOrg   float64 
 5   newbalanceOrig  float64 
 6   nameDest        string  
 7   oldbalanceDest  float64 
 8   newbalanceDest  float64 
 9   isFraud         int8    
 10  isFlaggedFraud  int8    
dtypes: category(1), float64(5), int32(1), int8(2), string(2)
memory usage: 382.3 MB


,count,mean,std,min,25%,50%,75%,max
step,"6,362,620.0000",243.3972,142.3320,1.0000,156.0000,239.0000,335.0000,743.0000
amount,"6,362,620.0000","179,861.9035","603,858.2315",0.0000,"13,389.5700","74,871.9400","208,721.4775","92,445,516.6400"
oldbalanceOrg,"6,362,620.0000","833,883.1041","2,888,242.6730",0.0000,0.0000,"14,208.0000","107,315.1750","59,585,040.3700"
newbalanceOrig,"6,362,620.0000","855,113.6686","2,924,048.5030",0.0000,0.0000,0.0000,"144,258.4100","49,585,040.3700"
oldbalanceDest,"6,362,620.0000","1,100,701.6665","3,399,180.1130",0.0000,0.0000,"132,705.6650","943,036.7075","356,015,889.3500"
newbalanceDest,"6,362,620.0000","1,224,996.3982","3,674,128.9421",0.0000,0.0000,"214,661.4400","1,111,909.2500","356,179,278.9200"
isFraud,"6,362,620.0000",0.0013,0.0359,0.0000,0.0000,0.0000,0.0000,1.0000
isFlaggedFraud,"6,362,620.0000",0.0000,0.0016,0.0000,0.0000,0.0000,0.0000,1.0000


,dtype,missing_count,missing_percent
step,int32,0,0.0000
type,category,0,0.0000
amount,float64,0,0.0000
nameOrig,string,0,0.0000
oldbalanceOrg,float64,0,0.0000
newbalanceOrig,float64,0,0.0000
nameDest,string,0,0.0000
oldbalanceDest,float64,0,0.0000
newbalanceDest,float64,0,0.0000
isFraud,int8,0,0.0000


Exact duplicate rows: 0
Negative transaction amounts: 0
isFraud values: [0, 1]
isFlaggedFraud values: [0, 1]


## 3. Fraud Class Balance

Fraud is expected to be rare. Report counts and percentages together; accuracy alone can be misleading for an imbalanced dataset.

In [5]:
class_summary = df.groupby('isFraud').size().rename('count').to_frame()
class_summary['percent'] = class_summary['count'] / len(df) * 100
display(class_summary)
fraud_rate = df['isFraud'].mean() * 100
print(f'Overall fraud rate: {fraud_rate:.4f}%')

ax = class_summary['count'].rename(index={0: 'Non-fraud', 1: 'Fraud'}).plot.bar(color=['steelblue', 'tomato'])
ax.set(title='Transaction Counts by Fraud Label', xlabel='Label', ylabel='Number of transactions')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()


,count,percent
isFraud,,
0,6354407,99.8709
1,8213,0.1291


Overall fraud rate: 0.1291%


C:\Users\lenovo\AppData\Local\Temp\ipykernel_18444\3072759874.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Transaction Types

Compare transaction volume and fraud rate separately. A common transaction type does not necessarily have a high fraud rate.

In [6]:
type_summary = df.groupby('type', observed=True).agg(
    transactions=('isFraud', 'size'),
    fraud_count=('isFraud', 'sum'),
    fraud_rate=('isFraud', 'mean'),
    median_amount=('amount', 'median'),
    total_amount=('amount', 'sum'),
).sort_values('transactions', ascending=False)
type_summary['fraud_rate_percent'] = type_summary.pop('fraud_rate') * 100
display(type_summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
type_summary['transactions'].plot.bar(ax=axes[0], color='steelblue')
axes[0].set(title='Transaction Volume by Type', xlabel='Type', ylabel='Transactions')
type_summary['fraud_rate_percent'].plot.bar(ax=axes[1], color='tomato')
axes[1].set(title='Fraud Rate by Type', xlabel='Type', ylabel='Fraud rate (%)')
plt.tight_layout()
plt.show()


,transactions,fraud_count,median_amount,total_amount,fraud_rate_percent
type,,,,,
CASH_OUT,2237500,4116,"147,072.1850","394,412,995,224.4900",0.1840
PAYMENT,2151495,0,"9,482.1900","28,093,371,138.3700",0.0000
CASH_IN,1399284,0,"143,427.7100","236,367,391,912.4600",0.0000
TRANSFER,532909,4097,"486,308.3900","485,291,987,263.1700",0.7688
DEBIT,41432,0,"3,048.9900","227,199,221.2800",0.0000


C:\Users\lenovo\AppData\Local\Temp\ipykernel_18444\1923705354.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Transaction Amounts

Amounts can have a long right tail. Review quantiles and use a log transformation for the histogram. Negative amounts, if present, are excluded from this plot and were counted in the quality check.

In [7]:
display(df.groupby('isFraud')['amount'].describe(percentiles=[0.5, 0.9, 0.99]).T)

fig, ax = plt.subplots()
for label, color in [(0, 'steelblue'), (1, 'tomato')]:
    amounts = df.loc[(df['isFraud'] == label) & (df['amount'] >= 0), 'amount']
    ax.hist(np.log1p(amounts), bins=60, density=True, alpha=0.5,
            label='Fraud' if label else 'Non-fraud', color=color)
ax.set(title='Transaction Amount Distribution', xlabel='log(1 + amount)', ylabel='Density')
ax.legend()
plt.tight_layout()
plt.show()


isFraud,0,1
count,"6,354,407.0000","8,213.0000"
mean,"178,197.0417","1,467,967.2991"
std,"596,236.9813","2,404,252.9472"
min,0.0100,0.0000
50%,"74,684.7200","441,423.4400"
90%,"364,373.4440","4,521,723.5120"
99%,"1,586,064.1734","10,000,000.0000"
max,"92,445,516.6400","10,000,000.0000"


C:\Users\lenovo\AppData\Local\Temp\ipykernel_18444\3332871121.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Activity Over Simulated Time

Aggregate by simulation step (hour). Changes in fraud rate should be read alongside transaction counts, especially in low-volume steps. These are simulated hours, not calendar timestamps.

In [8]:
hourly = df.groupby('step').agg(
    transactions=('isFraud', 'size'), fraud_count=('isFraud', 'sum'),
    fraud_rate=('isFraud', 'mean'),
)
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(hourly.index, hourly['transactions'], color='steelblue')
axes[0].set(title='Transaction Activity over Time', ylabel='Transactions')
axes[1].plot(hourly.index, hourly['fraud_rate'] * 100, color='tomato')
axes[1].set(xlabel='Simulation step (hour)', ylabel='Fraud rate (%)')
plt.tight_layout()
plt.show()


C:\Users\lenovo\AppData\Local\Temp\ipykernel_18444\2424315253.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Balances and Numeric Relationships

Correlation describes association, not causation. Exclude account identifiers and treat correlations with the rare fraud label carefully. Balance changes may contain information recorded after a transaction; assess feature availability before using them in a predictive model.

In [9]:
numeric_columns = ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
                   'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']
corr = df[numeric_columns].corr()
display(corr['isFraud'].sort_values(ascending=False).to_frame('correlation_with_fraud'))

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr.index)), corr.index)
ax.set_title('Numeric Correlation Matrix')
fig.colorbar(im, ax=ax, label='Pearson correlation')
plt.tight_layout()
plt.show()

# Sample only for visualization; all summary tables use the complete dataset.
sample = df.sample(n=min(30_000, len(df)), random_state=RANDOM_STATE)
fig, ax = plt.subplots()
for label, color in [(0, 'steelblue'), (1, 'tomato')]:
    subset = sample.loc[sample['isFraud'] == label]
    ax.scatter(subset['oldbalanceOrg'], subset['amount'], s=12, alpha=0.35,
               color=color, label='Fraud' if label else 'Non-fraud')
ax.set_xscale('symlog', linthresh=1)
ax.set_yscale('symlog', linthresh=1)
ax.set(title='Origin Balance vs. Amount (Random Sample)',
       xlabel='Origin balance before transaction (symlog)', ylabel='Amount (symlog)')
ax.legend()
plt.tight_layout()
plt.show()


,correlation_with_fraud
isFraud,1.0000
amount,0.0767
isFlaggedFraud,0.0441
step,0.0316
oldbalanceOrg,0.0102
newbalanceDest,0.0005
oldbalanceDest,-0.0059
newbalanceOrig,-0.0081


C:\Users\lenovo\AppData\Local\Temp\ipykernel_18444\869620383.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\lenovo\AppData\Local\Temp\ipykernel_18444\869620383.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Existing Fraud Flag

Compare the system flag with the fraud label. Precision measures how many flagged transactions are actually fraudulent; recall measures how many fraudulent transactions are flagged. These metrics evaluate the existing flag, not a trained model.

In [10]:
flag_table = pd.crosstab(df['isFraud'], df['isFlaggedFraud']).reindex(
    index=[0, 1], columns=[0, 1], fill_value=0)
display(flag_table.rename_axis(index='Actual fraud', columns='System flag'))
tn, fp = flag_table.loc[0, 0], flag_table.loc[0, 1]
fn, tp = flag_table.loc[1, 0], flag_table.loc[1, 1]
precision = tp / (tp + fp) if tp + fp else np.nan
recall = tp / (tp + fn) if tp + fn else np.nan
display(pd.Series({'precision': precision, 'recall': recall}, name='Existing flag metrics'))


System flag,0,1
Actual fraud,,
0,6354407,0
1,8197,16


precision   1.0000
recall      0.0019
Name: Existing flag metrics, dtype: float64

## 9. Key Findings and Next Steps

The following findings are calculated from the loaded data. They describe this synthetic dataset and should not be assumed to generalize to real financial transactions.

In [11]:
fraud_types = type_summary.loc[type_summary['fraud_count'] > 0].index.tolist()
print(f'Total transactions: {len(df):,}')
print(f'Fraudulent transactions: {df["isFraud"].sum():,} ({fraud_rate:.4f}%)')
print(f'Most frequent transaction type: {type_summary.index[0]}')
print(f'Types containing fraud: {", ".join(map(str, fraud_types)) or "None"}')
print(f'Missing values: {df.isna().sum().sum():,}')
print(f'Existing flag precision: {precision:.4f}; recall: {recall:.4f}')


Total transactions: 6,362,620
Fraudulent transactions: 8,213 (0.1291%)
Most frequent transaction type: CASH_OUT
Types containing fraud: CASH_OUT, TRANSFER


Missing values: 0
Existing flag precision: 1.0000; recall: 0.0019


**Suggested next steps**
- Investigate suspicious patterns within each transaction type.
- Confirm that candidate features are available at prediction time to avoid leakage.
- Use a chronological train/test split when evaluating future transaction detection; consider repeated accounts across splits.
- For modeling, compare precision, recall, and precision-recall AUC, and select a threshold based on operational costs.
- Apply any resampling only to the training data, and validate findings on representative real data.
